In [39]:
# ruff: noqa: T201, T203, ERA001, BLE001
from __future__ import annotations

import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import json
import logging

import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

from config.config import DATA_DIR, RUN_DIR
from config.runs_vars import DATASET_SUBSETS_MAP
from src.constants import PAIRS_SEPARATOR
from src.evaluate import (
    get_gt_pairs,
    get_predictions_with_gt,
    plot_usage_histograms,
    read_run_metrics_df,
    save_analysis_results,
    store_run_metrics_df,
)
from src.formatting import (
    format_gt_pairs_filepath,
    format_oracle_pairs_filepath,
    format_oracle_pairs_precomputed_dir,
    format_oracle_pairs_with_prompts_path,
    format_predictions_run_path,
    format_run_path,
    format_storing_pathes_from_run_path,
    format_subsets_ontologies_paths,
)
from src.prompts.prompts import (
    prompt_direct_entity,
    prompt_direct_entity_ontological,
    prompt_direct_entity_with_synonyms,
    prompt_sequential_hierarchy,
    prompt_sequential_hierarchy_ontological,
    prompt_sequential_hierarchy_with_synonyms,
)

pd.set_option("display.max_rows", None)
logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [48]:
num_samples = 100
filter_out_reduced_oracle_pairs = True
seed = 0
version = "v0"
output_data_template = {
    "version": "1.0",
    "description": "This file contains a few-shot learning dataset for testing input-output pairs.",
    "OpenAI": {},
}


In [49]:
prompt_functions = [
    prompt_direct_entity,
    prompt_direct_entity_ontological,
    prompt_direct_entity_with_synonyms,
    prompt_sequential_hierarchy,
    prompt_sequential_hierarchy_ontological,
    prompt_sequential_hierarchy_with_synonyms,
]


for dataset in DATASET_SUBSETS_MAP:
    for subset in DATASET_SUBSETS_MAP[dataset]:
        print(f"Processing dataset: {dataset}, subset: {subset}")
        oracle_pairs_prompts_dir = format_oracle_pairs_precomputed_dir(dataset, subset)
        oracle_pairs_prompts_file = format_oracle_pairs_with_prompts_path(None, oracle_pairs_prompts_dir, suffix="")
        promts_df = pd.read_csv(oracle_pairs_prompts_file, sep="|", header=None)
        promts_df.columns = ["Source", "Target", "Prompt"][: len(promts_df.columns)]

        if filter_out_reduced_oracle_pairs:
            rediced_oracle_pairs_prompts_dir = format_oracle_pairs_precomputed_dir(dataset, subset, suffix="_reduced")
            reduced_oracle_pairs_prompts_file = format_oracle_pairs_with_prompts_path(
                None, rediced_oracle_pairs_prompts_dir, suffix=""
            )
            reduced_promts_df = pd.read_csv(reduced_oracle_pairs_prompts_file, sep="|", header=None)
            print("Number of pairs in original data:", len(promts_df))
            print("Number of pairs in reduced data:", len(reduced_promts_df))
            reduced_promts_df.columns = ["Source", "Target", "Prompt"][: len(reduced_promts_df.columns)]

            reduced_pairs = set(zip(reduced_promts_df["Source"], reduced_promts_df["Target"]))
            promts_df = promts_df[
                promts_df.apply(lambda row: (row["Source"], row["Target"]) not in reduced_pairs, axis=1)
            ]

        gt_positive_pairs_df = get_gt_pairs(format_gt_pairs_filepath(dataset, subset))

        gt_pairs = set(zip(gt_positive_pairs_df["Source"], gt_positive_pairs_df["Target"]))
        promts_df["is_positive"] = promts_df.apply(lambda row: (row["Source"], row["Target"]) in gt_pairs, axis=1)
        present_df = promts_df[promts_df["is_positive"]].drop(columns=["is_positive"])
        absent_df = promts_df[~promts_df["is_positive"]].drop(columns=["is_positive"])

        if len(present_df) < num_samples or len(absent_df) < num_samples:
            print(
                f"Not enough pairs to sample {num_samples} positive and {num_samples} negative pairs. "
                f"Available positive pairs: {len(present_df)}, Available negative pairs: {len(absent_df)}"
            )
            num_samples = min(len(present_df), len(absent_df))

        present_df = present_df.sample(n=num_samples, random_state=seed)
        absent_df = absent_df.sample(n=num_samples, random_state=seed)

        output_data = output_data_template.copy()

        for prompt_fn in prompt_functions:
            prompt_name = prompt_fn.__name__
            oracle_pairs_prompts_file = format_oracle_pairs_with_prompts_path(prompt_fn, oracle_pairs_prompts_dir)
            promts_df = pd.read_csv(oracle_pairs_prompts_file, sep="|", header=None)
            promts_df.columns = ["Source", "Target", "Prompt"][: len(promts_df.columns)]

            positive_pairs = set(zip(present_df["Source"], present_df["Target"]))
            positive_pairs_prompts_df = promts_df[
                promts_df.apply(lambda row: (row["Source"], row["Target"]) in positive_pairs, axis=1)
            ].reset_index(drop=True)
            negative_pairs = set(zip(absent_df["Source"], absent_df["Target"]))

            negative_pairs_prompts_df = promts_df[
                promts_df.apply(lambda row: (row["Source"], row["Target"]) in negative_pairs, axis=1)
            ].reset_index(drop=True)
            len(positive_pairs_prompts_df), len(negative_pairs_prompts_df)

            output_data_template["OpenAI"][prompt_name] = []
            # append one positive prompt one negative one by one
            for i in range(len(positive_pairs_prompts_df)):
                output_data_template["OpenAI"][prompt_name].append(
                    {"input": positive_pairs_prompts_df.iloc[i]["Prompt"], "output": '{"answer": true}'}
                )
                output_data_template["OpenAI"][prompt_name].append(
                    {"input": negative_pairs_prompts_df.iloc[i]["Prompt"], "output": '{"answer": false}'}
                )

        save_path = DATA_DIR / "preparation" / f"{dataset}/{subset}/{version}.json"

        save_path.parent.mkdir(parents=True, exist_ok=True)
        with save_path.open("w") as f:
            json.dump(output_data_template, f, indent=4)


Processing dataset: bioml-2024, subset: ncit-doid
Number of pairs in original data: 1248
Number of pairs in reduced data: 856
Not enough pairs to sample 100 positive and 100 negative pairs. Available positive pairs: 347, Available negative pairs: 45
Processing dataset: bioml-2024, subset: omim-ordo
Number of pairs in original data: 1464
Number of pairs in reduced data: 399
Processing dataset: bioml-2024, subset: snomed-fma.body
Number of pairs in original data: 2081
Number of pairs in reduced data: 980
Processing dataset: bioml-2024, subset: snomed-ncit.neoplas
Number of pairs in original data: 1797
Number of pairs in reduced data: 1305
Processing dataset: bioml-2024, subset: snomed-ncit.pharm
Number of pairs in original data: 2081
Number of pairs in reduced data: 557
Processing dataset: largebio, subset: fma-nci
Number of pairs in original data: 1258
Number of pairs in reduced data: 885
Processing dataset: largebio, subset: fma-snomed
Number of pairs in original data: 5284
Number of p

(50, 50)